In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy
import pandas
import statsmodels.api as sm
import seaborn
import statsmodels.formula.api as smf 
import matplotlib.pyplot as plt

pandas.set_option('display.float_format', lambda x:'%.2f'%x)

In [ ]:
data = pandas.read_csv('/kaggle/input/nesarc-huit/nesarc.csv', low_memory=False)

In [ ]:
data['IDNUM'] =pandas.to_numeric(data['IDNUM'], errors='coerce')
data['TAB12MDX'] = pandas.to_numeric(data['TAB12MDX'], errors='coerce')
data['MAJORDEPLIFE'] = pandas.to_numeric(data['MAJORDEPLIFE'], errors='coerce')
data['NDSymptoms'] = pandas.to_numeric(data['NDSymptoms'], errors='coerce')
data['SOCPDLIFE'] = pandas.to_numeric(data['SOCPDLIFE'], errors='coerce')
data['S3AQ3C1'] = pandas.to_numeric(data['S3AQ3C1'], errors='coerce')
data['AGE'] =pandas.to_numeric(data['AGE'], errors='coerce')
data['SEX'] = pandas.to_numeric(data['SEX'], errors='coerce')
data['S3AQ3B1'] = pandas.to_numeric(data['S3AQ3B1'], errors='coerce')

In [ ]:
sub1=data[(data['AGE']<=25) & (data['CHECK321']==1) & (data['S3AQ3B1']==1)]

In [ ]:
def NICOTINEDEP (x):
   if x['TAB12MDX']==1:
      return 1
   else: 
      return 0
sub1['NICOTINEDEP'] = sub1.apply (lambda x: NICOTINEDEP (x), axis=1)

In [ ]:
# logistic regression 
lreg1 = smf.logit(formula = 'NICOTINEDEP ~ SOCPDLIFE', data = sub1).fit()
print (lreg1.summary())

In [ ]:
print ("ODD")
print (numpy.exp(lreg1.params))

In [ ]:
# odd ratios với 95% độ tin cậy
conf = lreg1.conf_int()
conf['ODD'] = lreg1.params
conf.columns = ['Lower CI', 'Upper CI', 'ODD']
print (numpy.exp(conf))

Giá trị ODD của Intercept = 1.46 nghĩa là, khi không có yếu tố chứng sợ xã hội (SOCPDLIFE = 0), tỷ lệ odds (tỷ số khả năng xảy ra và không xảy ra) của việc nghiện Nicotine là 1.46. Điều này có nghĩa là trong nhóm người không có chứng sợ xã hội, khả năng nghiện Nicotine của họ là 1.46 lần so với khả năng không nghiện.
Giá trị ODD của SOCPDLIFE = 3.43 cho thấy rằng, nếu một người có chứng sợ xã hội (SOCPDLIFE = 1), tỷ lệ odds của họ bị nghiện Nicotine sẽ cao hơn 3.43 lần so với người không có chứng sợ xã hội, khi các yếu tố khác giữ nguyên.

In [ ]:
# logistic regression with chứng sợ xã hội và trầm cảm
lreg2 = smf.logit(formula = 'NICOTINEDEP ~ SOCPDLIFE + MAJORDEPLIFE', data = sub1).fit()
print (lreg2.summary())

In [ ]:
# odd ratios với 95% khoảng tin cậy
params = lreg2.params
conf = lreg2.conf_int()
conf['ODD'] = params
conf.columns = ['Lower CI', 'Upper CI', 'ODD']
print (numpy.exp(conf))

* Khi không có cả chứng sợ xã hội và trầm cảm (cả SOCPDLIFE và MAJORDEPLIFE đều bằng 0), tỷ lệ odds của việc nghiện Nicotine là 1.10, tức là khả năng nghiện Nicotine gần như không thay đổi đáng kể (odds gần với 1, nghĩa là khả năng nghiện và không nghiện gần bằng nhau).
* Những người có chứng sợ xã hội (SOCPDLIFE = 1) có khả năng nghiện Nicotine cao hơn 2.31 lần so với những người không có chứng sợ xã hội, giữ các yếu tố khác không đổi (đặc biệt là trầm cảm). Khoảng tin cậy cho thấy mối liên hệ này có thể dao động từ 1.17 đến 4.57 lần, và vì khoảng tin cậy này không chứa 1, điều này chỉ ra rằng mối liên hệ này có ý nghĩa thống kê.
* Những người bị trầm cảm (MAJORDEPLIFE = 1) có khả năng nghiện Nicotine cao hơn 3.70 lần so với những người không bị trầm cảm, giữ các yếu tố khác không đổi (đặc biệt là chứng sợ xã hội). Khoảng tin cậy từ 2.74 đến 4.98 cho thấy mối liên hệ này rất mạnh và ý nghĩa về mặt thống kê (vì khoảng tin cậy không chứa 1).